# Phase 1 — Classical ML baseline

**Q-Care Detect** (research only)

> Research risk classification — not for clinical use.
>
> This project is for educational and research purposes only. It is not a medical device and must not be used for clinical diagnosis or treatment decisions.

This notebook walks through the same steps as `python -m src.train_classical`. The terminal command is the official way to save models and plots. Run cells in order from the **project root** (or set the working directory to the repo root).

We do **not** train a quantum model here.

## 1. Load the public dataset

sklearn's original labels are `0 = malignant`, `1 = benign`. We convert so **1 = malignant** (the positive class).

In [ ]:
import sys
from pathlib import Path

# Allow `import src` when the notebook kernel starts in notebooks/
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import load_breast_cancer_dataset, print_dataset_inspection

data = load_breast_cancer_dataset()
print_dataset_inspection(data)

## 2. Train/test split (no leakage)

80% train, 20% test, stratified, `random_state=42`. We split **before** scaling.

In [ ]:
from src.preprocessing import stratified_train_test_split

x_train, x_test, y_train, y_test = stratified_train_test_split(
    data.features,
    data.target,
)
print(len(x_train), "train rows,", len(x_test), "test rows")
print("Classes in train:", sorted(y_train.unique()))
print("Classes in test:", sorted(y_test.unique()))

## 3. Train two classical models

- **Logistic Regression** uses `StandardScaler` inside a Pipeline (fit on train only).
- **Random Forest** is not scaled (trees do not need it).

In [ ]:
from src.train_classical import train_and_save_models, predict_with_probabilities, LR_MODEL_NAME, RF_MODEL_NAME
from src.evaluate import beginner_summary, evaluate_models
from src.utils import RESEARCH_DISCLAIMER

models = train_and_save_models(x_train, y_train)
predictions = {}
probabilities = {}
for name, model in models.items():
    y_pred, y_prob = predict_with_probabilities(model, x_test)
    predictions[name] = y_pred
    probabilities[name] = y_prob

metrics = evaluate_models(y_test, predictions, probabilities, save_plots=True)
metrics

## 4. Read the beginner summary

These scores are for this public test split only. They are not a diagnosis.

In [ ]:
print(RESEARCH_DISCLAIMER)
print()
print(beginner_summary(metrics))